# 06 - Data Preprocessing & Augmentation

**Goal:** Prepare the PlantVillage dataset for AI model training by resizing, normalizing, converting to RGB, and applying data augmentation.

| Step | Operation |
|------|-----------|
| 1 | Resize → 224×224 |
| 2 | Convert → RGB |
| 3 | Normalize → 0 to 1 |
| 4 | Augmentation → Rotation, Flip, Zoom, Brightness |

| Info | Value |
|------|-------|
| Dataset | PlantVillage |
| Task | Multi-Crop Disease Detection |
| Phase | 6 — Data Preprocessing |

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATASET_PATH = "../dataset/PlantVillage"
IMAGE_SIZE   = (224, 224)

print("TensorFlow Version:", tf.__version__)
print("Target Image Size :", IMAGE_SIZE)

## Step 1 — Resize to 224×224 + RGB Conversion

In [ ]:
# Pick a random sample image
classes = [f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))]
rand_class = random.choice(classes)
class_path = os.path.join(DATASET_PATH, rand_class)
rand_img   = random.choice(os.listdir(class_path))
img_path   = os.path.join(class_path, rand_img)

# Original image
original = Image.open(img_path)

# Step 1: Convert to RGB
rgb_image = original.convert("RGB")

# Step 2: Resize to 224x224
resized = rgb_image.resize(IMAGE_SIZE)

# Show before/after
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(original)
axes[0].set_title(f"Original\n{original.size[0]}×{original.size[1]}  |  {original.mode}")
axes[0].axis("off")

axes[1].imshow(resized)
axes[1].set_title(f"Resized + RGB\n{resized.size[0]}×{resized.size[1]}  |  {resized.mode}")
axes[1].axis("off")

plt.suptitle(rand_class, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Original Size :", original.size)
print("Resized Size  :", resized.size)
print("Mode          :", resized.mode)

## Step 2 — Normalize Pixel Values (0–255 → 0–1)

In [ ]:
# Convert to numpy array
img_array = np.array(resized)

print("Before Normalization:")
print("  Shape :", img_array.shape)
print("  Min   :", img_array.min())
print("  Max   :", img_array.max())
print("  dtype :", img_array.dtype)

# Normalize: divide by 255
normalized = img_array / 255.0

print("\nAfter Normalization:")
print("  Shape :", normalized.shape)
print("  Min   :", round(normalized.min(), 4))
print("  Max   :", round(normalized.max(), 4))
print("  dtype :", normalized.dtype)
print("\n✅ Pixel values are now in range [0, 1]")

## Step 3 — Data Augmentation

Augmentation ලෙස කරන operations:
- **Rotation** — image ටික rotate කරනවා
- **Flip** — mirror image
- **Zoom** — crop in/out
- **Brightness** — light conditions vary කරනවා

In [ ]:
# Augmentation using TensorFlow / Keras
datagen = ImageDataGenerator(
    rescale            = 1.0 / 255,       # Normalize
    rotation_range     = 30,              # Rotate ±30°
    horizontal_flip    = True,            # Mirror flip
    vertical_flip      = False,
    zoom_range         = 0.2,             # Zoom 20%
    brightness_range   = [0.7, 1.3],      # Brightness vary
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    fill_mode          = "nearest"
)

print("✅ ImageDataGenerator configured with:")
print("   - Rescale        : 1/255")
print("   - Rotation       : ±30°")
print("   - Horizontal Flip: Yes")
print("   - Zoom           : 20%")
print("   - Brightness     : 0.7 – 1.3")
print("   - Width/Height Shift: 10%")

In [ ]:
# Visualize augmented versions of one image
img_array_4d = np.expand_dims(img_array, axis=0)  # Shape: (1, 224, 224, 3)

aug_gen = datagen.flow(img_array_4d, batch_size=1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f"Data Augmentation Preview\n{rand_class}", fontsize=13, fontweight="bold")

labels = ["Original", "Aug 1", "Aug 2", "Aug 3", "Aug 4", "Aug 5", "Aug 6", "Aug 7"]

# Show original in first cell
axes[0][0].imshow(resized)
axes[0][0].set_title("Original", fontsize=10)
axes[0][0].axis("off")

# Show augmented in remaining cells
for i, ax in enumerate(axes.flatten()[1:]):
    batch = next(aug_gen)
    aug_img = batch[0]
    ax.imshow(aug_img)
    ax.set_title(f"Augmented {i+1}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig("augmentation_preview.png", dpi=150, bbox_inches="tight")
plt.show()

print("✅ Augmentation preview saved as: augmentation_preview.png")

## Step 4 — Full Pipeline Summary

In [ ]:
# Final pipeline - ready for model training
train_datagen = ImageDataGenerator(
    rescale            = 1.0 / 255,
    rotation_range     = 30,
    horizontal_flip    = True,
    zoom_range         = 0.2,
    brightness_range   = [0.7, 1.3],
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    fill_mode          = "nearest",
    validation_split   = 0.2   # 80% train, 20% validation
)

val_datagen = ImageDataGenerator(
    rescale          = 1.0 / 255,
    validation_split = 0.2
)

print("=" * 50)
print("Preprocessing Pipeline Summary")
print("=" * 50)
print(f"Image Size     : {IMAGE_SIZE}")
print(f"Normalization  : 0–255  →  0–1")
print(f"RGB Conversion : Yes")
print(f"Augmentation   : Rotation, Flip, Zoom, Brightness")
print(f"Train Split    : 80%")
print(f"Val Split      : 20%")
print("=" * 50)
print("✅ Pipeline ready for Model Training!")